# Module 4: Online Evaluation & Continuous Improvement

## Overview

In this module, you'll complete the production lifecycle by:

1. **Configuring online evaluation** for continuous monitoring
2. **Simulating drift scenarios** to trigger degradation
3. **Analyzing failed samples** from traces
4. **Improving the agent** with updated prompts
5. **Validating improvements** with safe deployment practices

### Learning Objectives
1. Set up AgentCore online evaluation with sampling
2. Understand drift detection and analysis
3. Implement agent improvements based on failures
4. Use shadow testing and canary deployment concepts

### Time: ~60 minutes

## Step 1: Setup

In [ ]:
import json
import os
import sys
import boto3
import pandas as pd
from datetime import datetime
import time

# Load deployment info from Module 3
try:
    %store -r deployment_info
    %store -r runtime
    %store -r baseline_metrics
    print("Loaded deployment info from previous modules")
    print(f"Agent: {deployment_info.get('agent_name')}")
    print(f"Agent ID: {deployment_info.get('agent_id')}")
except:
    print("Could not load deployment info - ensure Module 3 completed")
    deployment_info = {}

session = boto3.Session()
REGION = session.region_name or 'us-east-1'
print(f"Region: {REGION}")

In [ ]:
# Load drift scenarios
with open('drift_scenarios.json', 'r') as f:
    drift_config = json.load(f)

print(f"Loaded {len(drift_config['scenarios'])} drift scenarios:")
for scenario in drift_config['scenarios']:
    print(f"  - {scenario['id']}: {scenario['name']} ({scenario['drift_type']})")

## Step 2: Configure Online Evaluation

Set up continuous evaluation that samples production traffic.

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation

# Initialize evaluation client
eval_client = Evaluation(region=REGION)

# Create online evaluation configuration
agent_id = deployment_info.get('agent_id')

if agent_id:
    print(f"Creating online evaluation config for agent: {agent_id}")
    
    try:
        online_config = eval_client.create_online_config(
            agent_id=agent_id,
            config_name="ecommerce_production_eval",
            sampling_rate=100,  # 100% for demo, use 10-20% in production
            evaluator_list=[
                "Builtin.GoalSuccessRate",
                "Builtin.Correctness",
                "Builtin.ToolSelectionAccuracy"
            ],
            config_description="Production monitoring for e-commerce customer service",
            auto_create_execution_role=True
        )
        print(f"Online evaluation config created: {online_config.get('onlineEvaluationConfigId')}")
    except Exception as e:
        print(f"Error creating config (may already exist): {e}")
else:
    print("No agent_id available - skipping online evaluation setup")
    print("Proceeding with manual evaluation simulation...")

## Step 3: Simulate Drift Scenarios

Run drift scenarios to trigger degradation and observe evaluation scores.

In [ ]:
def simulate_drift_scenario(runtime_or_service, scenario, verbose=True):
    """Run a drift scenario and collect results"""
    results = []
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Scenario: {scenario['name']}")
        print(f"Type: {scenario['drift_type']}")
        print(f"Description: {scenario['description']}")
        print(f"{'='*60}")
    
    for query in scenario['queries']:
        if verbose:
            print(f"\nQuery: {query}")
        
        try:
            start = datetime.now()
            
            # Try runtime invoke (production) or direct call (local)
            if hasattr(runtime_or_service, 'invoke'):
                response = runtime_or_service.invoke({"prompt": query})
                response_text = response.get('response', str(response))
            else:
                response_text = str(runtime_or_service.chat(query))
            
            latency = (datetime.now() - start).total_seconds() * 1000
            
            if verbose:
                print(f"Response: {response_text[:200]}...")
            
            results.append({
                'scenario_id': scenario['id'],
                'scenario_name': scenario['name'],
                'drift_type': scenario['drift_type'],
                'query': query,
                'response': response_text,
                'latency_ms': latency,
                'success': True,
                'timestamp': datetime.now().isoformat()
            })
            
        except Exception as e:
            if verbose:
                print(f"Error: {e}")
            results.append({
                'scenario_id': scenario['id'],
                'query': query,
                'response': None,
                'error': str(e),
                'success': False,
                'timestamp': datetime.now().isoformat()
            })
    
    return results

In [ ]:
# Run selected drift scenarios
# If runtime available, use it; otherwise use local service

all_drift_results = []

# Select scenarios to run
scenarios_to_run = [
    drift_config['scenarios'][0],  # New Product Category
    drift_config['scenarios'][1],  # Policy Change
    drift_config['scenarios'][2],  # Informal Language
    drift_config['scenarios'][5],  # Adversarial Input
]

print("Running drift scenarios...")
print(f"Total scenarios: {len(scenarios_to_run)}")

for scenario in scenarios_to_run:
    try:
        if 'runtime' in dir() and runtime:
            results = simulate_drift_scenario(runtime, scenario)
        else:
            # Fall back to local simulation
            sys.path.insert(0, '../01-multi-agent-prototype/agents')
            from orchestrator import MultiAgentCustomerService
            local_service = MultiAgentCustomerService(region=REGION)
            results = simulate_drift_scenario(local_service, scenario)
        
        all_drift_results.extend(results)
    except Exception as e:
        print(f"Error running scenario {scenario['id']}: {e}")

In [ ]:
# Analyze drift results
df_drift = pd.DataFrame(all_drift_results)

print("\nDrift Scenario Results Summary")
print("="*60)
print(f"Total queries: {len(df_drift)}")
print(f"Successful: {df_drift['success'].sum()}")
print(f"Average latency: {df_drift['latency_ms'].mean():.0f}ms")

print("\nBy Scenario:")
for scenario_id in df_drift['scenario_id'].unique():
    scenario_df = df_drift[df_drift['scenario_id'] == scenario_id]
    print(f"  {scenario_id}: {len(scenario_df)} queries, {scenario_df['success'].mean():.0%} success")

## Step 4: Evaluate Drift Scenario Responses

Use evaluators to score the responses and identify issues.

In [ ]:
# Import evaluators
sys.path.insert(0, '../02-evaluation-baseline')
from custom_evaluators import (
    RoutingAccuracyEvaluator,
    PolicyComplianceEvaluator,
    ResponseQualityEvaluator
)

# Initialize evaluators
routing_eval = RoutingAccuracyEvaluator()
policy_eval = PolicyComplianceEvaluator()
quality_eval = ResponseQualityEvaluator()

print("Evaluators initialized")

In [ ]:
# Evaluate drift responses
drift_evaluations = []

print("Evaluating drift scenario responses...\n")

for idx, row in df_drift.iterrows():
    if not row['success'] or not row['response']:
        continue
    
    print(f"Evaluating: {row['scenario_id']} - {row['query'][:40]}...")
    
    eval_result = {
        'scenario_id': row['scenario_id'],
        'scenario_name': row['scenario_name'],
        'drift_type': row['drift_type'],
        'query': row['query'],
        'response': row['response']
    }
    
    try:
        # Policy compliance (important for drift_002)
        policy_result = policy_eval.evaluate(
            input_text=row['query'],
            output_text=row['response']
        )
        eval_result['policy_score'] = policy_result.score
        eval_result['policy_reasoning'] = policy_result.reasoning
    except:
        eval_result['policy_score'] = None
    
    try:
        # Response quality
        quality_result = quality_eval.evaluate(
            input_text=row['query'],
            output_text=row['response']
        )
        eval_result['quality_score'] = quality_result.score
        eval_result['quality_reasoning'] = quality_result.reasoning
    except:
        eval_result['quality_score'] = None
    
    drift_evaluations.append(eval_result)

df_eval = pd.DataFrame(drift_evaluations)
print(f"\nEvaluated {len(df_eval)} responses")

In [ ]:
# Identify issues by scenario
print("\nDrift Detection Analysis")
print("="*60)

for scenario_id in df_eval['scenario_id'].unique():
    scenario_data = df_eval[df_eval['scenario_id'] == scenario_id]
    
    policy_avg = scenario_data['policy_score'].mean()
    quality_avg = scenario_data['quality_score'].mean()
    
    print(f"\n{scenario_id}: {scenario_data['scenario_name'].iloc[0]}")
    print(f"  Drift Type: {scenario_data['drift_type'].iloc[0]}")
    print(f"  Policy Compliance: {policy_avg:.2f}" if policy_avg else "  Policy Compliance: N/A")
    print(f"  Response Quality: {quality_avg:.2f}" if quality_avg else "  Response Quality: N/A")
    
    # Flag issues
    if policy_avg and policy_avg < 0.7:
        print(f"  [!] LOW POLICY COMPLIANCE - Needs prompt update")
    if quality_avg and quality_avg < 0.7:
        print(f"  [!] LOW RESPONSE QUALITY - Needs investigation")

## Step 5: Review Failed Samples

Analyze specific failures to understand root causes.

In [ ]:
# Find low-scoring responses
threshold = 0.7

low_quality = df_eval[df_eval['quality_score'] < threshold] if 'quality_score' in df_eval.columns else pd.DataFrame()
low_policy = df_eval[df_eval['policy_score'] < threshold] if 'policy_score' in df_eval.columns else pd.DataFrame()

print("Failed Samples Analysis")
print("="*60)

print(f"\nLow Quality Responses (< {threshold}):")
if len(low_quality) > 0:
    for _, row in low_quality.iterrows():
        print(f"\n  Scenario: {row['scenario_name']}")
        print(f"  Query: {row['query']}")
        print(f"  Score: {row['quality_score']:.2f}")
        print(f"  Response: {row['response'][:150]}...")
        if row.get('quality_reasoning'):
            print(f"  Issue: {row['quality_reasoning'][:200]}")
else:
    print("  No low quality responses found")

print(f"\nLow Policy Compliance (< {threshold}):")
if len(low_policy) > 0:
    for _, row in low_policy.iterrows():
        print(f"\n  Scenario: {row['scenario_name']}")
        print(f"  Query: {row['query']}")
        print(f"  Score: {row['policy_score']:.2f}")
        if row.get('policy_reasoning'):
            print(f"  Issue: {row['policy_reasoning'][:200]}")
else:
    print("  No policy compliance issues found")

## Step 6: Implement Improvements

Apply improved prompts based on identified issues.

In [ ]:
from improved_prompts import get_improved_prompts, get_prompt_changes_summary

print("Prompt Improvement Summary")
print("="*60)
print(get_prompt_changes_summary())

In [ ]:
# Compare V1 vs V2 prompts on drift scenarios
improved_prompts = get_improved_prompts()

print("\nKey Improvements Made:")
print("-"*40)
print("1. Policy Update: 30 days -> 45 days return window")
print("2. New status: Added 'backordered' handling")
print("3. Informal language: Added normalization guidance")
print("4. Out-of-scope: Explicit list of unsupported categories")
print("5. Adversarial: Added prompt injection protection")
print("6. Urgency: Added realistic promise guidelines")

## Step 7: Validate Improvements (Conceptual Shadow Testing)

In production, you would deploy V2 in shadow mode to compare with V1.

In [ ]:
# Conceptual shadow testing workflow
print("\nShadow Testing Workflow (Conceptual)")
print("="*60)

shadow_workflow = """
1. DEPLOY V2 AS SHADOW
   - Deploy improved agent with tag 'shadow'
   - Shadow receives same traffic as V1 but doesn't serve responses
   
   +------------------------------------------+
   |           Customer Request               |
   +-------------------+-----------------------+
                       |
          +------------+------------+
          |                         |
          v                         v
   +-------------+           +-------------+
   |   V1 (Prod) |           | V2 (Shadow) |
   |   Serves    |           |   Silent    |
   +------+------+           +------+------+
          |                         |
          v                         v
   +-------------+           +-------------+
   |  Response   |           |   Logged    |
   |  to User    |           |   for Eval  |
   +-------------+           +-------------+

2. COMPARE METRICS
   - Evaluate both V1 and V2 responses
   - Compare: goal_success, quality, policy compliance
   - Statistical significance test

3. PROMOTE IF BETTER
   - If V2 metrics significantly better -> proceed to canary
   - If V2 regresses -> investigate and iterate
"""
print(shadow_workflow)

In [ ]:
# Simulate V1 vs V2 comparison
print("\nSimulated V1 vs V2 Comparison")
print("="*60)

# Use baseline as V1, simulate V2 improvements
v1_metrics = {
    'goal_success': baseline_metrics.get('goal_success_rate', 0.75),
    'policy_compliance': baseline_metrics.get('policy_compliance', 0.70),
    'response_quality': baseline_metrics.get('response_quality', 0.72)
}

# V2 expected improvements (conservative estimates)
v2_metrics = {
    'goal_success': min(v1_metrics['goal_success'] + 0.10, 0.95),
    'policy_compliance': min(v1_metrics['policy_compliance'] + 0.15, 0.95),
    'response_quality': min(v1_metrics['response_quality'] + 0.08, 0.95)
}

print(f"\n{'Metric':<25} {'V1':<10} {'V2':<10} {'Delta':<10}")
print("-"*55)
for metric in v1_metrics:
    delta = v2_metrics[metric] - v1_metrics[metric]
    indicator = '[+]' if delta > 0 else '[!]'
    print(f"{metric:<25} {v1_metrics[metric]:<10.2%} {v2_metrics[metric]:<10.2%} {delta:+.2%} {indicator}")

print("\n[+] V2 shows improvement across all metrics")
print("   Recommendation: Proceed to canary deployment")

## Step 8: Canary Deployment (Conceptual)

In [ ]:
print("\nCanary Deployment Strategy")
print("="*60)

canary_strategy = """
Stage 1: 10% traffic -> V2 (30 min)
  - Monitor: goal_success >= 80%, error_rate < 5%
  - Rollback if: goal_success < 75% OR error_rate > 10%

Stage 2: 25% traffic -> V2 (60 min)
  - Monitor: all metrics, customer satisfaction
  - Rollback if: any metric 10% below V1

Stage 3: 50% traffic -> V2 (2 hours)
  - Full A/B comparison
  - Statistical significance check

Stage 4: 100% traffic -> V2
  - V1 kept warm for 24h for emergency rollback

Automated Rollback Triggers:
  - goal_success_rate < 0.75
  - error_rate > 0.05
  - latency_p99 > 5000ms
  - safety_violation > 0
"""
print(canary_strategy)

In [ ]:
# Simulate canary rollout
import time

print("\nSimulating Canary Rollout...")
print("="*60)

stages = [
    {'stage': 1, 'traffic': 10, 'duration': 'simulated'},
    {'stage': 2, 'traffic': 25, 'duration': 'simulated'},
    {'stage': 3, 'traffic': 50, 'duration': 'simulated'},
    {'stage': 4, 'traffic': 100, 'duration': 'complete'}
]

for stage in stages:
    print(f"\nStage {stage['stage']}: {stage['traffic']}% traffic to V2")
    
    # Simulate metrics (improving slightly each stage)
    sim_goal = v2_metrics['goal_success'] - (4 - stage['stage']) * 0.02
    sim_error = 0.02 + (4 - stage['stage']) * 0.005
    
    print(f"  Goal Success: {sim_goal:.2%}")
    print(f"  Error Rate: {sim_error:.2%}")
    
    # Check rollback conditions
    if sim_goal < 0.75:
        print("  [!] ROLLBACK: Goal success below threshold")
        break
    elif sim_error > 0.05:
        print("  [!] ROLLBACK: Error rate too high")
        break
    else:
        print("  [OK] Metrics healthy - proceeding")
    
    time.sleep(0.5)  # Simulate time passing

print("\n" + "="*60)
print("CANARY DEPLOYMENT SUCCESSFUL!")
print("V2 is now serving 100% of production traffic")

## Summary

In this module, you completed the production lifecycle:

1. **Configured online evaluation** with sampling-based monitoring
2. **Simulated drift scenarios** including:
   - New product categories (knowledge gaps)
   - Policy changes (stale information)
   - Informal language (input distribution shift)
   - Adversarial inputs (security)
3. **Analyzed failed samples** using custom evaluators
4. **Implemented improvements** with V2 prompts:
   - Updated policies
   - Better informal language handling
   - Explicit out-of-scope guidance
   - Adversarial input protection
5. **Validated with safe deployment** (shadow + canary concepts)

### Complete MLOps Lifecycle

```
+------------------------------------------------------------------+
|                    CONTINUOUS IMPROVEMENT LOOP                    |
|                                                                   |
|   +---------+    +---------+    +---------+    +---------+       |
|   | Monitor |==>| Detect  |==>| Analyze |==>| Improve |       |
|   | (Online |    | Drift   |    | Failures|    | Agent   |       |
|   |  Eval)  |    |         |    |         |    |         |       |
|   +---------+    +---------+    +---------+    +----+----+       |
|        ^                                            |            |
|        |         +---------+    +---------+        |            |
|        +---------| Deploy  |<===|Validate |<========+            |
|                  | (Canary)|    | (Shadow)|                      |
|                  +---------+    +---------+                      |
|                                                                  |
+------------------------------------------------------------------+
```

### Key Takeaways

1. **Online evaluation** catches issues before customers complain
2. **Drift scenarios** help test robustness proactively
3. **Failed sample analysis** reveals specific improvement opportunities
4. **Safe deployment** (shadow + canary) reduces production risk
5. **Continuous improvement** is essential for production agents

## Workshop Complete!

You have successfully completed the E-Commerce Customer Service Multi-Agent Workshop!

### What You Built
- Multi-agent system with cost-optimized LLM strategy (Sonnet 4.5 + Haiku 4.5)
- Comprehensive evaluation with synthetic datasets
- Production deployment on AgentCore Runtime
- Continuous monitoring and improvement pipeline

### Next Steps
- Customize for your use case
- Add more specialized agents
- Implement actual shadow/canary infrastructure
- Connect to real customer data sources